# Search Images by other Images

## Calculate crop embeddings

In [ ]:
import json

from os import listdir, makedirs, path
from PIL import Image as PImage, ImageOps as PImageOps

from models.SigLip2 import SigLip2

In [ ]:
CROP_MODEL = "all"
CROP_IMGS_PATH = f"../../imgs/arts/crops_{CROP_MODEL}"
CROP_EMBED_PATH = f"./metadata/json/art-crops/embeddings/{CROP_MODEL}"

# CROP_IMGS_PATH = "../../imgs/palms/crops"
# CROP_EMBED_PATH = "./metadata/json/palm-crops/embeddings"

makedirs(CROP_EMBED_PATH, exist_ok=True)

model = SigLip2()
model_name = type(model).__name__.lower()
print(model_name)

crop_fnames = sorted([fn for fn in listdir(CROP_IMGS_PATH) if fn.endswith(".jpg")])
print(len(crop_fnames))

In [ ]:
for idx,fname in enumerate(crop_fnames):
  if idx % 100 == 0:
    print(f"{idx} / {len(crop_fnames)}")

  qid = fname.replace(".jpg", "")
  image_path = path.join(CROP_IMGS_PATH, fname)
  embedding_path = path.join(CROP_EMBED_PATH, f"{qid}.json")

  embeds = {}
  if path.isfile(embedding_path):
    with open(embedding_path, "r") as ifp:
      embeds = json.load(ifp)[qid]

  if model_name in embeds:
    continue

  img = PImageOps.exif_transpose(PImage.open(image_path).convert("RGB"))
  embeds[model_name] = [round(v, 8) for v in model.get_image_embedding(img).tolist()]
  embedding_data = { qid: embeds }

  with open(embedding_path, "w") as ofp:
    json.dump(embedding_data, ofp, separators=(",",":"), sort_keys=True, ensure_ascii=False)

### Export crop embeddings to file

In [ ]:
import json
from os import listdir, path

DATA_PREFIX = "20260801"
JSON_DIR = "./metadata/json"
CROP_MODEL = "all"
EMBEDS_DIR = path.join(JSON_DIR, "art-crops", "embeddings", f"{CROP_MODEL}")

emb_files = sorted([f for f in listdir(EMBEDS_DIR) if f.endswith("json")])

sig_embs = {}
for ef in emb_files:
  qid = ef.replace(".json", "")
  with open(path.join(EMBEDS_DIR, ef), "r") as ifp:
    ed = json.load(ifp)
    sig_embs[qid] = ed[qid]["siglip2"]

with open(path.join(JSON_DIR, f"{DATA_PREFIX}_crops_{CROP_MODEL}.json"), "w") as ofp:
  json.dump(sig_embs, ofp, separators=(",",":"), sort_keys=True, ensure_ascii=False)